# Examen práctico de Estadística Descriptiva (EA2)
## Matrícula de Educación Superior, Región de Tarapacá (2021)

**Tema:** correlación y regresión (Ítem 1) · modalidad y probabilidades (Ítem 2)

## 0. Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

## 1. Datos, población y variables

Se trabaja con el archivo `02_MATRICULAS_ED_SUPERIOR_TARAPACA_2021.xlsx`, que contiene **15.459 registros y 28 variables** sobre matrícula de educación superior en la Región de Tarapacá para 2021. La población de interés son los estudiantes matriculados en instituciones de educación superior de esa región durante ese año; el análisis describe los registros disponibles en la base y no presupone que sean una muestra aleatoria.

**Unidad de observación:** cada fila se interpreta como un registro de matrícula asociado a una carrera e institución. Por ello, el número de filas no se debe interpretar automáticamente como el número de personas únicas.

**Variables:** la base reúne características de los estudiantes y sus matrículas (por ejemplo, género, edad y año de ingreso), datos de institución y carrera (tipo, nivel, área, modalidad, jornada y duración), y montos de matrícula y arancel. El diccionario de la sección 1.1 identifica las variables utilizadas y su tipo.

**Alcance:** el año 2021 corresponde al período de la base. `AÑO INGRESO` indica cuándo ingresó el estudiante a la carrera, por lo que sus valores pueden ser anteriores a 2021; no representa una serie anual de matrículas nuevas.

In [5]:
baseDatos = "02_MATRICULAS_ED_SUPERIOR_TARAPACA_2021.xlsx"
df = pd.read_excel(baseDatos)
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15459 entries, 0 to 15458
Data columns (total 28 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   ID                                       15459 non-null  int64  
 1   GENERO                                   15459 non-null  object 
 2   EDAD                                     15459 non-null  int64  
 3   RANGO EDAD                               15459 non-null  object 
 4   AÑO INGRESO                              15459 non-null  int64  
 5   SEMESTRE INGRESO                         15459 non-null  object 
 6   TIPO DE INSTITUCION                      15459 non-null  object 
 7   NOMBRE DE INSTITUCION                    15459 non-null  object 
 8   ACREDITACION INSTITUCIONAL               15459 non-null  object 
 9   PERIODO DE ACREDITACION                  14066 non-null  object 
 10  AÑOS DE ACREDITACION                     14066

### 1.1 Diccionario de variables utilizadas

| Variable | Tipo | Escala | Uso en el trabajo |
|---|---|---|---|
| `VALOR MATRICULA (PESOS)` | Cuantitativa continua | Razón | Ítem 1.1 |
| `VALOR ARANCEL (PESOS)` | Cuantitativa continua | Razón | Ítem 1.1, 1.2 |
| `DURACION TOTAL CARRERA (SEMESTRES)` | Cuantitativa discreta | Razón | Ítem 1.2 |
| `DURACION PLAN DE ESTUDIO (SEMESTRES)` | Cuantitativa discreta | Razón | _opcional_ |
| `TIPO DE INSTITUCION` | Cualitativa nominal | Nominal | Ítem 1.1, 2.2 |
| `AREA CONOCIMIENTO` | Cualitativa nominal | Nominal | Ítem 1.2, 2.3 |
| `MODALIDAD` | Cualitativa nominal | Nominal | Ítem 2.1, 2.2 |
| `JORNADA` | Cualitativa nominal | Nominal | Ítem 2.1 (apoyo) |
| `AÑO INGRESO` | Cuantitativa discreta | Intervalo | Ítem 2.1 |
| `GENERO` | Cualitativa nominal | Nominal | Ítem 2.3 |
| `NIVEL CARRERA` / `NIVEL DE ESTUDIO CARRERA` | Cualitativa ordinal | Ordinal | Filtros |

### 1.2 Calidad de datos

In [6]:
# Verificar que existan todas las columnas clave
cols = [
    "VALOR MATRICULA (PESOS)", "VALOR ARANCEL (PESOS)",
    "DURACION TOTAL CARRERA (SEMESTRES)", "TIPO DE INSTITUCION",
    "AREA CONOCIMIENTO", "MODALIDAD", "JORNADA", "AÑO INGRESO", "GENERO",
    "NIVEL DE ESTUDIO CARRERA", "NIVEL CARRERA",
]
faltan = [c for c in cols if c not in df.columns]
assert not faltan, f"Columnas no encontradas: {faltan}"

# Tabla de valores nulos y ceros por variable clave
display(pd.DataFrame({
    "nulos": df[cols].isna().sum(),
    "ceros": (df[cols] == 0).sum(),
}))

# Contar filas duplicadas
print("Filas duplicadas:", df.duplicated().sum())

,nulos,ceros
VALOR MATRICULA (PESOS),0,61
VALOR ARANCEL (PESOS),0,10
DURACION TOTAL CARRERA (SEMESTRES),0,0
TIPO DE INSTITUCION,0,0
AREA CONOCIMIENTO,0,0
MODALIDAD,0,0
JORNADA,0,0
AÑO INGRESO,0,0
GENERO,0,0
NIVEL DE ESTUDIO CARRERA,0,0


Filas duplicadas: 0


### 1.3 Decisiones de limpieza / filtrado

Se definió una única base de análisis (`df_clean`) con estas reglas:

- **Nivel de estudio:** se conservan solo registros de pregrado (`NIVEL DE ESTUDIO CARRERA == "Pregrado"`), para mantener un alcance académico común y no mezclar niveles de formación distintos.
- **Montos:** se conservan solo registros con matrícula y arancel mayores que cero. Los valores cero se excluyen porque el análisis de costos compara montos positivos; no se consideran datos nulos ni se afirma que sean errores. La revisión de calidad detectó 61 registros con matrícula o arancel igual a cero (61 con matrícula cero y 10 con arancel cero).
- **Nulos y duplicados:** las variables clave no presentan valores nulos y no se detectaron filas duplicadas; por ello no se aplican exclusiones adicionales por esos motivos.

Con las reglas combinadas, la base pasa de **15.459** a **15.321 registros**; se excluyen **138** registros en total. Como los criterios pueden coincidir en una misma fila, las exclusiones no se suman por separado. Todos los análisis que usen `df_clean` quedan limitados a matrículas de pregrado con ambos montos positivos; las conclusiones no describen los registros excluidos.

In [7]:
# Limpieza y filtracion de datos
df_clean = df[
    (df["NIVEL DE ESTUDIO CARRERA"] == "Pregrado")
    & (df["VALOR MATRICULA (PESOS)"] > 0)
    & (df["VALOR ARANCEL (PESOS)"] > 0)
].copy()
print(f"n original = {len(df):,} | n analizado = {len(df_clean):,}")

# ¿Dónde están los ceros?
display(df[(df["VALOR MATRICULA (PESOS)"] == 0) | (df["VALOR ARANCEL (PESOS)"] == 0)]["TIPO DE INSTITUCION"].value_counts())

n original = 15,459 | n analizado = 15,321


,count
TIPO DE INSTITUCION,
Universidades CRUCH,46
Universidades Privadas,15


---
# ÍTEM 1 · Correlación y regresión

## Ítem 1.1 Correlación matrícula–arancel (global y por tipo de institución)

**Pregunta:** Estudie la correlación entre el valor de la matrícula y el del arancel para toda la base y luego por tipo de institución. ¿Los grupos se comportan parecido? ¿Vale la pena estudiar todos los grupos?

### Correlación global

In [ ]:
# Calcular la correlación de Pearson y de Spearman entre matrícula y arancel, con n y p-valor.

### Correlación por tipo de institución

In [ ]:
# Calcular, para cada tipo de institución, la correlación (Pearson y Spearman) y el n. Mostrar en una tabla.

### Visualización

In [ ]:
# Gráfico de dispersión matrícula vs arancel: uno global y uno por tipo de institución, con recta de ajuste.

### ¿Vale la pena estudiar todos los grupos?

In [ ]:
# Tabla por grupo con n, variabilidad y cantidad de valores distintos de matrícula, para decidir si vale la pena analizarlo.

#### Interpretación

_Redactar aquí._

## Ítem 1.2 Relación arancel–duración de la carrera

**Pregunta:** ¿Cuál es la relación entre arancel y duración? ¿Es verdad que mientras más larga la carrera, más cara? ¿Es igual en todas las áreas del conocimiento?

### Análisis global

In [ ]:
# Dispersión arancel vs duración con recta de regresión. Reportar pendiente, intercepto, R² y correlación.

### Análisis por área del conocimiento

In [ ]:
# Tabla por área del conocimiento con correlación, pendiente, R² y n. Excluir áreas con n insuficiente, justificando.

### Visualización por área

In [ ]:
# Gráficos de dispersión arancel vs duración, uno por área del conocimiento, con su recta de regresión.

### Diagnóstico del modelo

In [ ]:
# Revisar los residuos del modelo: linealidad, homocedasticidad y valores atípicos influyentes.

#### Interpretación

_Redactar aquí._

---
# ÍTEM 2 · Modalidad y probabilidades

## Ítem 2.1 ¿Vuelta a la presencialidad o avance de formatos no presenciales?

**Pregunta:** ¿Se produjo una gran vuelta a la presencialidad o los formatos que incluyen no presencialidad han ganado terreno?

### Distribución global de modalidad

In [ ]:
# Tabla de frecuencias absolutas y relativas de la modalidad.

### Evolución por año de ingreso

In [ ]:
# Tabla cruzada año de ingreso × modalidad, con porcentajes por año.

### Visualización

In [ ]:
# Gráfico de barras apiladas al 100 % (o líneas) con la modalidad por año de ingreso.

#### Interpretación

_Redactar aquí._

## Ítem 2.2 Tipo de institución con mayor uso de formatos no exclusivamente presenciales

**Pregunta:** ¿Qué tipo de institución emplea de mayor manera formatos no exclusivamente presenciales?

### Tabla cruzada tipo × modalidad

In [ ]:
# Tabla cruzada tipo de institución × modalidad, con porcentajes por fila y n.

### Visualización

In [ ]:
# Gráfico de barras horizontales con el % de matrícula no exclusivamente presencial por tipo de institución.

### Asociación estadística (opcional)

In [ ]:
# Contraste de asociación entre tipo de institución y modalidad (chi-cuadrado) y medida de intensidad (V de Cramér).

#### Interpretación

_Redactar aquí._

## Ítem 2.3 Probabilidades: área de Educación según género

**Pregunta:** ¿Cuál es la probabilidad de seleccionar, entre los estudiantes hombres, a uno del área de Educación? ¿Y entre las mujeres? ¿Hay discrepancia aparente? ¿Cómo se explica?

### Tabla de contingencia género × área

In [ ]:
# Tabla de contingencia género × (Educación / otras áreas), con totales.

### Probabilidades condicionales

In [ ]:
# Calcular P(E|H), P(E|M) y P(E) a partir de la tabla de contingencia, mostrando conteos y fórmulas.

### Comparación e independencia

In [ ]:
# Comparar P(E|H) y P(E|M) (diferencia y razón) y contrastar independencia con chi-cuadrado.

### Explicación de la discrepancia

In [ ]:
# Analizar la composición por género dentro de cada tipo de institución, nivel u otra variable para explicar la discrepancia. Incluir un gráfico.

#### Interpretación

_Redactar aquí._